In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
fund_master = pd.read_csv("../data/raw/01_fund_master.csv")

nav_history = pd.read_csv("../data/raw/02_nav_history.csv")

aum = pd.read_csv("../data/raw/03_aum_by_fund_house.csv")

monthly_sip = pd.read_csv("../data/raw/04_monthly_sip_inflows.csv")

category_inflows = pd.read_csv("../data/raw/05_category_inflows.csv")

folio_count = pd.read_csv("../data/raw/06_industry_folio_count.csv")

performance = pd.read_csv("../data/raw/07_scheme_performance.csv")

transactions = pd.read_csv("../data/raw/08_investor_transactions.csv")

holdings = pd.read_csv("../data/raw/09_portfolio_holdings.csv")

benchmark = pd.read_csv("../data/raw/10_benchmark_indices.csv")

In [3]:
nav_history["date"] = pd.to_datetime(
    nav_history["date"]
)

nav_history = nav_history.sort_values(
    ["amfi_code", "date"]
)

nav_history = nav_history.drop_duplicates()

nav_history["nav"] = (
    nav_history
    .groupby("amfi_code")["nav"]
    .ffill()
)

invalid_nav = (
    nav_history["nav"] <= 0
).sum()

print("Invalid NAV:", invalid_nav)

Invalid NAV: 0


In [4]:
transactions["transaction_date"] = pd.to_datetime(
    transactions["transaction_date"]
)

In [5]:
valid_types = [
    "SIP",
    "Lumpsum",
    "Redemption"
]

invalid_types = transactions[
    ~transactions["transaction_type"]
    .isin(valid_types)
]

print("Invalid Types:", len(invalid_types))

Invalid Types: 0


In [6]:
valid_kyc = [
    "Verified",
    "Pending"
]

invalid_kyc = transactions[
    ~transactions["kyc_status"]
    .isin(valid_kyc)
]

print("Invalid KYC:", len(invalid_kyc))

Invalid KYC: 0


In [7]:
invalid_amount = (
    transactions["amount_inr"] <= 0
).sum()

print("Invalid Amounts:", invalid_amount)

Invalid Amounts: 0


In [8]:
expense_anomalies = performance[
    (performance["expense_ratio_pct"] < 0.1)
    |
    (performance["expense_ratio_pct"] > 2.5)
]

print(
    "Expense Ratio Anomalies:",
    len(expense_anomalies)
)

Expense Ratio Anomalies: 0


In [9]:
return_cols = [
    "return_1yr_pct",
    "return_3yr_pct",
    "return_5yr_pct"
]

performance[return_cols].describe()

,return_1yr_pct,return_3yr_pct,return_5yr_pct
count,40.000000,40.000000,40.000000
mean,14.376000,14.089000,14.516750
std,4.883023,4.617253,4.454021
min,4.260000,5.140000,5.430000
25%,11.735000,12.035000,12.340000
50%,14.620000,14.205000,14.185000
75%,16.392500,15.882500,17.585000
max,24.930000,23.390000,23.800000


In [10]:
aum["date"] = pd.to_datetime(aum["date"])
aum = aum.drop_duplicates()

In [11]:
monthly_sip = monthly_sip.drop_duplicates()

In [12]:
category_inflows = category_inflows.drop_duplicates()

In [13]:
folio_count = folio_count.drop_duplicates()

In [14]:
holdings = holdings.drop_duplicates()

In [15]:
benchmark["date"] = pd.to_datetime(
    benchmark["date"]
)

benchmark = benchmark.drop_duplicates()

In [16]:
processed_path = Path("../data/processed")

processed_path.mkdir(
    exist_ok=True,
    parents=True
)

In [17]:
fund_master.to_csv(
    processed_path / "fund_master_cleaned.csv",
    index=False
)

nav_history.to_csv(
    processed_path / "nav_history_cleaned.csv",
    index=False
)

aum.to_csv(
    processed_path / "aum_by_fund_house_cleaned.csv",
    index=False
)

monthly_sip.to_csv(
    processed_path / "monthly_sip_inflows_cleaned.csv",
    index=False
)

category_inflows.to_csv(
    processed_path / "category_inflows_cleaned.csv",
    index=False
)

folio_count.to_csv(
    processed_path / "industry_folio_count_cleaned.csv",
    index=False
)

performance.to_csv(
    processed_path / "scheme_performance_cleaned.csv",
    index=False
)

transactions.to_csv(
    processed_path / "investor_transactions_cleaned.csv",
    index=False
)

holdings.to_csv(
    processed_path / "portfolio_holdings_cleaned.csv",
    index=False
)

benchmark.to_csv(
    processed_path / "benchmark_indices_cleaned.csv",
    index=False
)

print("All cleaned files saved.")

All cleaned files saved.


In [20]:
date_dim = pd.DataFrame(
    pd.date_range(
        start=nav_history["date"].min(),
        end=nav_history["date"].max(),
        freq="D"
    ),
    columns=["date"]
)

date_dim["year"] = date_dim["date"].dt.year
date_dim["quarter"] = date_dim["date"].dt.quarter
date_dim["month"] = date_dim["date"].dt.month
date_dim["month_name"] = date_dim["date"].dt.month_name()
date_dim["weekday"] = date_dim["date"].dt.day_name()

date_dim.head()

,date,year,quarter,month,month_name,weekday
0,2022-01-03,2022,1,1,January,Monday
1,2022-01-04,2022,1,1,January,Tuesday
2,2022-01-05,2022,1,1,January,Wednesday
3,2022-01-06,2022,1,1,January,Thursday
4,2022-01-07,2022,1,1,January,Friday


In [21]:
date_dim.to_csv(
    "../data/processed/dim_date.csv",
    index=False
)

print("dim_date.csv saved")

dim_date.csv saved


# Day 2 Cleaning Summary

## NAV History
- Date conversion completed
- Sorting completed
- Duplicate removal completed
- NAV validation completed
- Forward fill completed

## Investor Transactions
- Date conversion completed
- Transaction type validation completed
- Amount validation completed
- KYC validation completed

## Scheme Performance
- Return metrics validated
- Expense ratio validation completed

## Output
10 cleaned datasets generated in data/processed